# QDArchive - Part 1: Data Acquisition
**Repositories assigned:** Zenodo (Repo #1) · Harvard Dataverse (Repo #10)

This notebook covers:
1. Exploring and testing search queries for both repositories
2. Setting up the SQLite metadata database schema
3. A download pipeline skeleton for QDA files and associated data

---
> **Data principle:** Do not transform or clean data at download time. Store raw values; quality fixing is a separate step.

## 0: Setup & Dependencies

In [1]:
# Install any missing packages (safe to re-run)
# %pip install requests sqlite-utils tqdm

import requests
import sqlite3
import json
import os
import time
from pathlib import Path
from datetime import datetime, timezone
from pprint import pprint

# ── Root folder where all downloaded files will live ──────────────────────────
DOWNLOAD_ROOT = Path("downloads")
DB_PATH       = Path("qdarchive_metadata.db")

DOWNLOAD_ROOT.mkdir(exist_ok=True)
(DOWNLOAD_ROOT / "zenodo").mkdir(exist_ok=True)
(DOWNLOAD_ROOT / "harvard-dataverse").mkdir(exist_ok=True)

print("Folder structure ready:", list(DOWNLOAD_ROOT.iterdir()))

Folder structure ready: [PosixPath('downloads/harvard-dataverse'), PosixPath('downloads/zenodo')]


---
## 1: SQLite Schema

Schema follows the spec in `SQLite_Meta_Data_Database_Schema.xlsx` exactly.

Tables:
- **REPOSITORIES** - master list of the 20 assigned repos
- **PROJECTS** - one row per research project found
- **FILES** - one row per file belonging to a project
- **KEYWORDS** - one raw keyword per row (no splitting or cleaning yet)
- **PERSON_ROLE** - contributors and their role enum
- **LICENSES** - licenses linked to a project

In [2]:
def create_schema(db_path: Path) -> sqlite3.Connection:
    """Create all tables. Safe to call multiple times (IF NOT EXISTS)."""
    con = sqlite3.connect(db_path)
    con.row_factory = sqlite3.Row
    cur = con.cursor()

    # ── REPOSITORIES ─────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS repositories (
        id              INTEGER PRIMARY KEY,
        name            TEXT    NOT NULL,
        base_url        TEXT    NOT NULL,
        api_url         TEXT,
        notes           TEXT
    )""")

    # ── PROJECTS ─────────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS projects (
        id                          INTEGER PRIMARY KEY AUTOINCREMENT,
        query_string                TEXT,
        repository_id               INTEGER NOT NULL REFERENCES repositories(id),
        repository_url              TEXT    NOT NULL,
        project_url                 TEXT    NOT NULL UNIQUE,
        version                     TEXT,
        title                       TEXT    NOT NULL,
        description                 TEXT    NOT NULL,
        language                    TEXT,           -- BCP 47 e.g. 'en-US'
        doi                         TEXT,
        upload_date                 DATE,
        download_date               TIMESTAMP NOT NULL,
        download_repository_folder  TEXT    NOT NULL,
        download_project_folder     TEXT    NOT NULL,
        download_version_folder     TEXT,
        download_method             TEXT    NOT NULL  -- 'SCRAPING' | 'API-CALL'
            CHECK(download_method IN ('SCRAPING','API-CALL'))
    )""")

    # ── FILES ─────────────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS files (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id  INTEGER NOT NULL REFERENCES projects(id),
        file_name   TEXT    NOT NULL,
        file_type   TEXT    NOT NULL,   -- just the extension e.g. 'qdpx'
        status      TEXT    NOT NULL
            CHECK(status IN (
                'SUCCEEDED',
                'FAILED_SERVER_UNRESPONSIVE',
                'FAILED_LOGIN_REQUIRED',
                'FAILED_TOO_LARGE'
            ))
    )""")

    # ── KEYWORDS ──────────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS keywords (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id  INTEGER NOT NULL REFERENCES projects(id),
        keyword     TEXT    NOT NULL
        -- Raw keyword string; one per row; no splitting/cleaning at download time
    )""")

    # ── PERSON_ROLE ───────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS person_role (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id  INTEGER NOT NULL REFERENCES projects(id),
        name        TEXT    NOT NULL,
        role        TEXT    NOT NULL
            CHECK(role IN ('UPLOADER','AUTHOR','OWNER','OTHER','UNKNOWN'))
    )""")

    # ── LICENSES ──────────────────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS licenses (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id  INTEGER NOT NULL REFERENCES projects(id),
        license_str TEXT    NOT NULL  -- Raw string from repo; clean later
    )""")

    # ── Seed the two assigned repositories ───────────────────────────────────
    cur.executemany("""
    INSERT OR IGNORE INTO repositories(id, name, base_url, api_url, notes)
    VALUES (?, ?, ?, ?, ?)""", [
        (1,  "Zenodo",           "https://zenodo.org",
              "https://zenodo.org/api",
              "CERN open research repository; InvenioRDM-based REST API"),
        (10, "Harvard Dataverse","https://dataverse.harvard.edu",
              "https://dataverse.harvard.edu/api",
              "Dataverse v6 Search API; no API key needed for public records"),
    ])

    con.commit()
    print(f"Schema ready at '{db_path}'")
    return con


con = create_schema(DB_PATH)

Schema ready at 'qdarchive_metadata.db'


In [3]:
# ── Verify tables were created ────────────────────────────────────────────────
tables = con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables:", [r["name"] for r in tables])

# Check seeded repos
repos = con.execute("SELECT id, name, api_url FROM repositories").fetchall()
for r in repos:
    print(f"  Repo {r['id']:>2}: {r['name']} → {r['api_url']}")

Tables: ['repositories', 'projects', 'sqlite_sequence', 'files', 'keywords', 'person_role', 'licenses']
  Repo  1: Zenodo → https://zenodo.org/api
  Repo 10: Harvard Dataverse → https://dataverse.harvard.edu/api


---
## 2: Zenodo: API Exploration & Query Testing

**API base:** `https://zenodo.org/api/records`

Key parameters:
| Parameter | Meaning |
|---|---|
| `q` | Elasticsearch/Lucene query string |
| `type` | Filter by type: `dataset`, `software`, etc. |
| `access_right` | `open` to restrict to open-access only |
| `page` / `size` | Pagination (max `size=100`) |
| `sort` | `mostrecent` \| `bestmatch` |

No API token needed for public records.

In [4]:
ZENODO_API = "https://zenodo.org/api/records"

def zenodo_search(query: str, size: int = 5, page: int = 1) -> dict:
    """Run a Zenodo record search and return the raw JSON response."""
    params = {
        "q":            query,
        "type":         "dataset",
        "access_right": "open",
        "size":         size,
        "page":         page,
        "sort":         "bestmatch",
    }
    resp = requests.get(ZENODO_API, params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


def print_zenodo_hits(data: dict, max_items: int = 5):
    """Print a quick summary of search hits."""
    hits = data.get("hits", {})
    total = hits.get("total", 0)
    if isinstance(total, dict):
        total = total.get("value", 0)
    print(f"Total results: {total}")
    print("-" * 60)
    for rec in hits.get("hits", [])[:max_items]:
        meta  = rec.get("metadata", {})
        files = rec.get("files", [])
        exts  = {f["key"].rsplit(".", 1)[-1].lower() for f in files if "." in f["key"]}
        print(f"[{rec['id']}] {meta.get('title', '?')[:70]}")
        print(f"  DOI: {meta.get('doi','—')}  |  license: {meta.get('license',{}).get('id','—')}")
        print(f"  File extensions: {exts or '(no files listed)'}")
        print()

In [5]:
# ── Query 1: Direct QDA file extension hunt ───────────────────────────────────
# Best signal: records that actually contain a .qdpx file
q1 = 'qdpx'
r1 = zenodo_search(q1, size=5)
print(f"Query: '{q1}'")
print_zenodo_hits(r1)

Query: 'qdpx'
Total results: 5
------------------------------------------------------------
[16082705] Supporting Data for Scoping Review on Interlanguage Pragmatics in EFL 
  DOI: 10.5281/zenodo.16082705  |  license: cc-by-4.0
  File extensions: {'docx', 'xlsx', 'qdpx'}

[14535515] Polish debates on higher education in the Sejm and party manifestos
  DOI: 10.5281/zenodo.14535515  |  license: odc-by
  File extensions: {'mx22', 'pdf', 'mx20', 'qdpx'}

[18196304] Implementation Science, Program Fidelity, and Program Delivery in Scal
  DOI: 10.5281/zenodo.18196304  |  license: cc-by-4.0
  File extensions: {'csv', 'xlsx', 'qdpx'}

[17360366] PLANET4B Business Models Dataset Pisa [UNIPI] 20251014 v2
  DOI: 10.5281/zenodo.17360366  |  license: cc-by-4.0
  File extensions: {'docx', 'pdf', 'qdpx'}

[17347915] PLANET4B Frame Analysis Dataset Pisa [UNIPI] 20251013 v2
  DOI: 10.5281/zenodo.17347915  |  license: cc-by-4.0
  File extensions: {'docx', 'pdf', 'doc', 'qdpx'}



In [6]:
# ── Query 2: MaxQDA files ─────────────────────────────────────────────────────
q2 = 'mx24 OR mqda OR mx22'
r2 = zenodo_search(q2, size=5)
print(f"Query: '{q2}'")
print_zenodo_hits(r2)

Query: 'mx24 OR mqda OR mx22'
Total results: 3
------------------------------------------------------------
[17384976] Qualitative Data of the EduChallenge Project
  DOI: 10.5281/zenodo.17384976  |  license: cc-by-4.0
  File extensions: {'zip', 'mx24', 'xlsx'}

[7886149] Dataset for the Paper: "Security Defect Detection via Code Review: A S
  DOI: 10.5281/zenodo.7886149  |  license: cc-by-4.0
  File extensions: {'zip'}

[18002402] Replication Package for the Paper: "An Insight into Security Code Revi
  DOI: 10.5281/zenodo.18002402  |  license: cc-by-4.0
  File extensions: {'zip'}



In [7]:
# ── Query 3: NVivo / ATLAS.ti / QDA Miner extensions ─────────────────────────
q3 = 'nvp OR nvpx OR atlasproj OR hpr7 OR ppj OR pprj'
r3 = zenodo_search(q3, size=5)
print(f"Query: '{q3}'")
print_zenodo_hits(r3)

Query: 'nvp OR nvpx OR atlasproj OR hpr7 OR ppj OR pprj'
Total results: 13
------------------------------------------------------------
[3974105] Plasma HIV-RNA during week -144 - week 144
  DOI: 10.5281/zenodo.3974105  |  license: cc-by-4.0
  File extensions: {'xlsx'}

[4999969] Data from: Ambulatory versus inpatient management of severe nausea and
  DOI: 10.5061/dryad.c3g48  |  license: cc-zero
  File extensions: {'xlsx'}

[8116208] OCS fluxes from a coastal Antarctic tundra and soils measured by in si
  DOI: 10.5281/zenodo.8116208  |  license: cc-by-4.0
  File extensions: {'xlsx'}

[14630254] Taxas_Suicidio_Racas_Brasil_2018_2022
  DOI: 10.5281/zenodo.14630254  |  license: cc-by-4.0
  File extensions: {'xlsx', 'py', 'md'}

[10479096] Confocal microscopy dataset: Effect of glutamine starvation and NVP-BE
  DOI: 10.5281/zenodo.10479096  |  license: cc-by-4.0
  File extensions: {'zip'}



In [8]:
# ── Query 4: Phrase-based qualitative research data ───────────────────────────
q4 = '"qualitative research data" AND (interview OR transcript)'
r4 = zenodo_search(q4, size=5)
print(f"Query: '{q4}'")
print_zenodo_hits(r4)

Query: '"qualitative research data" AND (interview OR transcript)'
Total results: 1
------------------------------------------------------------
[12564823] WAKAF BERTEMPOH (MUAQQAT) SATU CADANGAN PENGIKTIRAFAN, PENGUKURAN DAN 
  DOI: 10.21474/IJAR01/18846  |  license: cc-by-4.0
  File extensions: {'pdf'}



In [9]:
# ── Query 5: QDA software keywords in title ───────────────────────────────────
q5 = 'metadata.title:(qualitative) AND metadata.title:(MAXQDA OR NVivo OR "ATLAS.ti" OR QDAcity)'
r5 = zenodo_search(q5, size=5)
print(f"Query: '{q5}'")
print_zenodo_hits(r5)

Query: 'metadata.title:(qualitative) AND metadata.title:(MAXQDA OR NVivo OR "ATLAS.ti" OR QDAcity)'
Total results: 4
------------------------------------------------------------
[18824484] Nvivo Data (qualitative network visualizations)
  DOI: 10.5281/zenodo.18824484  |  license: cc-by-4.0
  File extensions: {'png', 'jpg'}

[17559863] Qualitative Codebook for IoT Adoption in Nanostores: Thematic Framewor
  DOI: 10.5281/zenodo.17559863  |  license: cc-by-4.0
  File extensions: {'xlsx'}

[16418052] Atlas.ti file - Qualitative data analysis of ENGO reports and transcri
  DOI: 10.5281/zenodo.16418052  |  license: cc-by-4.0
  File extensions: {'atlasti'}

[15574364] S3 Appendix C: Detailed procedures for qualitative data handling and t
  DOI: 10.5281/zenodo.15574364  |  license: cc-by-4.0
  File extensions: {'docx'}



In [10]:
# ── Query 6: Broader interview study approach ─────────────────────────────────
q6 = '"interview study" AND (transcript OR coding OR thematic)'
r6 = zenodo_search(q6, size=5)
print(f"Query: '{q6}'")
print_zenodo_hits(r6)

Query: '"interview study" AND (transcript OR coding OR thematic)'
Total results: 9
------------------------------------------------------------
[7745146] Understanding Requirements Engineering Debt: Study Material
  DOI: 10.5281/zenodo.7745146  |  license: cc-by-4.0
  File extensions: {'zip'}

[15012123] Replication Package - How to Use Vision Videos: Validating a Framework
  DOI: 10.5281/zenodo.15012123  |  license: cc-by-4.0
  File extensions: {'pdf'}

[10870991] The public part of the interview extractions as thematic codings: used
  DOI: 10.5281/zenodo.10870991  |  license: cc-by-4.0
  File extensions: {'xlsx'}

[8322656] Data from: Co-creation in fully remote software teams
  DOI: 10.5061/dryad.z612jm6hw  |  license: cc-zero
  File extensions: {'xlsx', 'pdf', 'md'}

[18220176] Replication Package for "Smells Depend on the Context: An Interview St
  DOI: 10.5281/zenodo.18220176  |  license: cc-by-4.0
  File extensions: {'zip'}



In [11]:
# ── Inspect a single record in full ──────────────────────────────────────────
# Pick the first hit from query 1 (qdpx) and look at its structure
sample_hits = r1.get("hits", {}).get("hits", [])
if sample_hits:
    print("=== Sample Zenodo record structure ===")
    sample = sample_hits[0]
    pprint({
        "id":       sample.get("id"),
        "links":    sample.get("links", {}),
        "metadata": sample.get("metadata", {}),
        "files":    sample.get("files", [])[:3],   # first 3 files
    })
else:
    print("No hits found — check your network or try a different query")

=== Sample Zenodo record structure ===
{'files': [{'checksum': 'md5:a3984adebd752ada6e289c5698b7ffdf',
            'id': 'd3a86d27-9a32-414e-b505-eeb9a40d4677',
            'key': 'EFL_ILP_ScopingReview_CodedData_20250718.qdpx',
            'links': {'self': 'https://zenodo.org/api/records/16082705/files/EFL_ILP_ScopingReview_CodedData_20250718.qdpx/content'},
            'size': 70748772},
           {'checksum': 'md5:082bb24657b747327545004dc4c0bdf7',
            'id': '66c8987d-42a6-4995-8f26-31a6f6bfec53',
            'key': 'Theme_year.xlsx',
            'links': {'self': 'https://zenodo.org/api/records/16082705/files/Theme_year.xlsx/content'},
            'size': 19364},
           {'checksum': 'md5:6d6083f9b1a2ac5de7050ade10925fcf',
            'id': '457558f6-66ff-4fc3-88f3-c69dceb6ff08',
            'key': 'S2_Table. Data Charting and CCAT Scores of 56 '
                   'Articles.docx',
            'links': {'self': 'https://zenodo.org/api/records/16082705/files/S2_Table. '

### Zenodo: Helper functions to extract metadata into DB-ready dicts

In [12]:
def zenodo_record_to_project(record: dict, query_string: str) -> dict:
    """Map a Zenodo API record dict to the PROJECTS table schema."""
    meta  = record.get("metadata", {})
    rec_id = record["id"]

    # ── Person roles ─────────────────────────────────────────────────────────
    creators = meta.get("creators", [])
    persons = [
        {"name": c.get("name", "Unknown"), "role": "AUTHOR"}
        for c in creators
    ]

    # ── License ──────────────────────────────────────────────────────────────
    lic = meta.get("license", {}).get("id", None)

    # ── Keywords (raw strings; may be comma-separated — don't split yet) ─────
    keywords = meta.get("keywords", [])

    return {
        "query_string":               query_string,
        "repository_id":              1,
        "repository_url":             "https://zenodo.org",
        "project_url":                f"https://zenodo.org/records/{rec_id}",
        "version":                    meta.get("version"),
        "title":                      meta.get("title", ""),
        "description":                meta.get("description", ""),
        "language":                   meta.get("language"),
        "doi":                        meta.get("doi"),
        "upload_date":                meta.get("publication_date"),
        "download_date":              datetime.now(timezone.utc).isoformat(),
        "download_repository_folder": "zenodo",
        "download_project_folder":    str(rec_id),
        "download_version_folder":    meta.get("version"),
        "download_method":            "API-CALL",
        # Extra fields not in the flat project row — returned separately
        "_persons":   persons,
        "_keywords":  keywords,
        "_license":   lic,
        "_files":     record.get("files", []),
    }


# Test with the first qdpx hit
if sample_hits:
    proj_dict = zenodo_record_to_project(sample_hits[0], query_string="qdpx")
    pprint({k: v for k, v in proj_dict.items() if not k.startswith("_")})

{'description': '<p>This dataset contains coded data (QDPX), article summary '
                '(Word), and year-theme matrix (Excel) used in the scoping '
                'review.</p>',
 'doi': '10.5281/zenodo.16082705',
 'download_date': '2026-04-06T20:13:38.134545+00:00',
 'download_method': 'API-CALL',
 'download_project_folder': '16082705',
 'download_repository_folder': 'zenodo',
 'download_version_folder': 'v1.0',
 'language': 'eng',
 'project_url': 'https://zenodo.org/records/16082705',
 'query_string': 'qdpx',
 'repository_id': 1,
 'repository_url': 'https://zenodo.org',
 'title': 'Supporting Data for Scoping Review on Interlanguage Pragmatics in '
          'EFL Contexts',
 'upload_date': '2025-07-18',
 'version': 'v1.0'}


---
## 3:Harvard Dataverse: API Exploration & Query Testing

**Search endpoint:** `https://dataverse.harvard.edu/api/search`

Key parameters:
| Parameter | Meaning |
|---|---|
| `q` | Free-text or fielded query |
| `type` | `dataset` \| `dataverse` \| `file` |
| `per_page` | Results per page (max 1000) |
| `start` | Offset for pagination |
| `sort` | Field to sort by |
| `order` | `asc` \| `desc` |

No API key needed for public records. To get file metadata, use
`/api/datasets/{id}/versions/1.0/files`.

In [13]:
HDV_API  = "https://dataverse.harvard.edu/api"
HDV_SEARCH = f"{HDV_API}/search"

def hdv_search(query: str, per_page: int = 5, start: int = 0) -> dict:
    """Run a Harvard Dataverse dataset search."""
    params = {
        "q":        query,
        "type":     "dataset",
        "per_page": per_page,
        "start":    start,
        "sort":     "name",
        "order":    "asc",
    }
    resp = requests.get(HDV_SEARCH, params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


def print_hdv_hits(data: dict, max_items: int = 5):
    """Print a summary of Harvard Dataverse search results."""
    items = data.get("data", {}).get("items", [])
    total = data.get("data", {}).get("total_count", 0)
    print(f"Total results: {total}")
    print("-" * 60)
    for item in items[:max_items]:
        print(f"[{item.get('global_id','?')}] {item.get('name','?')[:70]}")
        print(f"  Published: {item.get('published_at','—')}  |  type: {item.get('type','—')}")
        print(f"  URL: {item.get('url','—')}")
        authors = item.get('authors', [])
        print(f"  Authors: {', '.join(authors[:3])}" + (" ..." if len(authors) > 3 else ""))
        print()

In [14]:
# ── Query 1: .qdpx extension ──────────────────────────────────────────────────
hq1 = 'qdpx'
hr1 = hdv_search(hq1, per_page=5)
print(f"Query: '{hq1}'")
print_hdv_hits(hr1)

Query: 'qdpx'
Total results: 2
------------------------------------------------------------
[doi:10.21950/PDWDPG] COMPREV(P)CANCER Base Datos del análisis de los comentarios sobre bron
  Published: 2025-11-13T01:00:12Z  |  type: dataset
  URL: https://doi.org/10.21950/PDWDPG
  Authors: Strategic Health Communication, GEAC

[doi:10.5683/SP3/OOCMMZ] Methane Emissions from the Global Oil and Gas Industry: A Scoping Revi
  Published: 2025-11-22T07:02:15Z  |  type: dataset
  URL: https://doi.org/10.5683/SP3/OOCMMZ
  Authors: Vollrath, Coleman



In [15]:
# ── Query 2: QDA software family ──────────────────────────────────────────────
hq2 = 'MAXQDA OR NVivo OR "ATLAS.ti"'
hr2 = hdv_search(hq2, per_page=5)
print(f"Query: '{hq2}'")
print_hdv_hits(hr2)

Query: 'MAXQDA OR NVivo OR "ATLAS.ti"'
Total results: 239
------------------------------------------------------------
[doi:10.3886/ICPSR38543] A Qualitative Assessment of Post-Partum Screening After Gestational Di
  Published: 2025-07-29T22:41:04Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38543
  Authors: Herrick, Cynthia

[doi:10.3886/ICPSR38543.V1] A Qualitative Assessment of Post-Partum Screening After Gestational Di
  Published: 2023-02-27T14:17:25Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38543.V1
  Authors: Herrick, Cynthia

[doi:10.3886/ICPSR38543.V2] A Qualitative Assessment of Post-Partum Screening After Gestational Di
  Published: 2023-02-27T14:20:39Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38543.V2
  Authors: Herrick, Cynthia

[doi:10.6141/TW-SRDA-E10217-1] A Study on the Relationship among the Principals' Space Leadership, Le
  Published: 2017-08-01T00:00:00Z  |  type: dataset
  URL: https://doi.org/10.6141/TW-SRDA-E10217-1
  Authors:

In [16]:
# ── Query 3: Qualitative interview data ───────────────────────────────────────
hq3 = '"qualitative" AND "interview" AND "transcript"'
hr3 = hdv_search(hq3, per_page=5)
print(f"Query: '{hq3}'")
print_hdv_hits(hr3)

Query: '"qualitative" AND "interview" AND "transcript"'
Total results: 395
------------------------------------------------------------
[doi:10.3886/ICPSR38668] A Community Partnership with Asian Pacific AIDS Intervention Team to E
  Published: 2023-02-06T14:24:48Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38668
  Authors: Tobin, Karin E.

[doi:10.3886/ICPSR38668.V1] A Community Partnership with Asian Pacific AIDS Intervention Team to E
  Published: 2023-02-06T14:24:46Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38668.V1
  Authors: Tobin, Karin E.

[doi:10.3886/ICPSR36446] A Multi-Method, Multi-Site Study of Gang Desistance, United States, 20
  Published: 2025-02-10T16:56:39Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR36446
  Authors: Esbensen, Finn-Aage

[doi:10.3886/ICPSR36446.V1] A Multi-Method, Multi-Site Study of Gang Desistance, United States, 20
  Published: 2022-04-24T03:24:08Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR36446.V1
  Au

In [17]:
# ── Query 4: Thematic analysis / grounded theory signal words ────────────────
hq4 = '"thematic analysis" OR "grounded theory" OR "content analysis"'
hr4 = hdv_search(hq4, per_page=5)
print(f"Query: '{hq4}'")
print_hdv_hits(hr4)

Query: '"thematic analysis" OR "grounded theory" OR "content analysis"'
Total results: 696
------------------------------------------------------------
[doi:10.7910/DVN/ADLFOS] "Do It For The Instagram" A Story of Gender and Social Status
  Published: 2016-07-22T12:24:04Z  |  type: dataset
  URL: https://doi.org/10.7910/DVN/ADLFOS
  Authors: Tilberry, Stephanie 

[doi:10.7910/DVN/2K3SKC] A Case Study of Video Blogs Over COVID-19
  Published: 2021-09-22T07:48:52Z  |  type: dataset
  URL: https://doi.org/10.7910/DVN/2K3SKC
  Authors: Cui, Jie

[doi:10.3886/ICPSR38668] A Community Partnership with Asian Pacific AIDS Intervention Team to E
  Published: 2023-02-06T14:24:48Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38668
  Authors: Tobin, Karin E.

[doi:10.3886/ICPSR38668.V1] A Community Partnership with Asian Pacific AIDS Intervention Team to E
  Published: 2023-02-06T14:24:46Z  |  type: dataset
  URL: https://doi.org/10.3886/ICPSR38668.V1
  Authors: Tobin, Karin E.

[doi:10.79

In [18]:
# ── Query 5: Broader all-extension sweep ──────────────────────────────────────
hq5 = 'nvp OR atlasproj OR mx24 OR mx22 OR f4p'
hr5 = hdv_search(hq5, per_page=5)
print(f"Query: '{hq5}'")
print_hdv_hits(hr5)

Query: 'nvp OR atlasproj OR mx24 OR mx22 OR f4p'
Total results: 7
------------------------------------------------------------
[doi:10.7910/DVN/2B7UDT] A5208: NNRTI vs PI Regimens for HIV Infected Women After They Have Tak
  Published: 2026-02-09T18:09:06Z  |  type: dataset
  URL: https://doi.org/10.7910/DVN/2B7UDT
  Authors: ACTG Statistical and Data Analysis Center

[doi:10.7910/DVN/NYCWFQ] A5208: NNRTI vs PI Regimens for HIV Infected Women After They Have Tak
  Published: 2026-02-09T16:25:05Z  |  type: dataset
  URL: https://doi.org/10.7910/DVN/NYCWFQ
  Authors: ACTG Statistical and Data Analysis Center

[doi:10.5064/F6V5VGX3] Data for: Improving Abortion Underreporting in the United States: A Co
  Published: 2024-05-02T20:13:43Z  |  type: dataset
  URL: https://doi.org/10.5064/F6V5VGX3
  Authors: Mueller, Jennifer, Kirstein, Marielle, VandeVusse, Alicia ...

[doi:10.5683/SP3/OOCMMZ] Methane Emissions from the Global Oil and Gas Industry: A Scoping Revi
  Published: 2025-11-22T07:02

In [19]:
# ── Get file list for a specific Harvard Dataverse dataset ───────────────────
# Replace DOI/persistent ID as needed
def hdv_get_files(persistent_id: str) -> list:
    """Return list of files for a dataset given its persistent ID (doi:...)."""
    url = f"{HDV_API}/datasets/:persistentId/versions/:latest/files"
    params = {"persistentId": persistent_id}
    resp = requests.get(url, params=params, timeout=20)
    resp.raise_for_status()
    return resp.json().get("data", [])


# Try with a real result from above — grab the first hit's global_id
hdv_items = hr1.get("data", {}).get("items", [])
if hdv_items:
    pid = hdv_items[0].get("global_id")
    print(f"Fetching files for: {pid}")
    try:
        files = hdv_get_files(pid)
        for f in files[:5]:
            df = f.get("dataFile", {})
            print(f"  {df.get('filename','?')} ({df.get('filesize','?')} bytes)  "
                  f"md5={df.get('md5','—')}")
    except Exception as e:
        print(f"Could not fetch files: {e}")
else:
    print("No HDV hits to inspect yet")

Fetching files for: doi:10.21950/PDWDPG
  COMENTARIOS_TIKTOK.qdpx (-1 bytes)  md5=unknown
  readme-es_tiktok.txt (-1 bytes)  md5=unknown


### Harvard Dataverse: metadata extraction helper

In [20]:
def hdv_item_to_project(item: dict, query_string: str) -> dict:
    """Map a Harvard Dataverse search result item to the PROJECTS table schema."""
    # global_id looks like 'doi:10.7910/DVN/XXXXX'
    global_id = item.get("global_id", "")
    doi_url   = f"https://doi.org/{global_id.replace('doi:','',1)}" if global_id.startswith("doi:") else None

    # Derive a folder-safe project ID from the last part of the DOI
    folder_id = global_id.split("/")[-1] if "/" in global_id else global_id

    authors = item.get("authors", [])
    persons = [
        {"name": a, "role": "AUTHOR"}
        for a in authors
    ]

    return {
        "query_string":               query_string,
        "repository_id":              10,
        "repository_url":             "https://dataverse.harvard.edu",
        "project_url":                item.get("url", ""),
        "version":                    str(item.get("versionId", "")) or None,
        "title":                      item.get("name", ""),
        "description":                item.get("description", ""),
        "language":                   None,   # not always available at search level
        "doi":                        doi_url,
        "upload_date":                item.get("published_at", "")[:10] if item.get("published_at") else None,
        "download_date":              datetime.now(timezone.utc).isoformat(),
        "download_repository_folder": "harvard-dataverse",
        "download_project_folder":    folder_id,
        "download_version_folder":    None,
        "download_method":            "API-CALL",
        "_persons":  persons,
        "_keywords": item.get("keywords", []),
        "_license":  item.get("license", None),
        "_global_id":global_id,
    }


# Test
if hdv_items:
    sample_proj = hdv_item_to_project(hdv_items[0], query_string="qdpx")
    pprint({k: v for k, v in sample_proj.items() if not k.startswith("_")})

{'description': 'Base de datos y análisis en el software Atlas.ti de '
                'comentarios relacionados con el bronceado y la fotoprevención '
                'solar en jóvenes que fueron descargados de publicaciones en '
                'la red social TikTok.',
 'doi': 'https://doi.org/10.21950/PDWDPG',
 'download_date': '2026-04-06T20:13:42.289868+00:00',
 'download_method': 'API-CALL',
 'download_project_folder': 'PDWDPG',
 'download_repository_folder': 'harvard-dataverse',
 'download_version_folder': None,
 'language': None,
 'project_url': 'https://doi.org/10.21950/PDWDPG',
 'query_string': 'qdpx',
 'repository_id': 10,
 'repository_url': 'https://dataverse.harvard.edu',
 'title': 'COMPREV(P)CANCER Base Datos del análisis de los comentarios sobre '
          'bronceado fotoprotección y fotoprevención solar en la red social '
          'TikTok',
 'upload_date': '2025-11-13',
 'version': '563183'}


---
## 4: Database Insert Helpers

In [21]:
INSERT_PROJECT = """
INSERT OR IGNORE INTO projects (
    query_string, repository_id, repository_url, project_url,
    version, title, description, language, doi,
    upload_date, download_date,
    download_repository_folder, download_project_folder,
    download_version_folder, download_method
) VALUES (
    :query_string, :repository_id, :repository_url, :project_url,
    :version, :title, :description, :language, :doi,
    :upload_date, :download_date,
    :download_repository_folder, :download_project_folder,
    :download_version_folder, :download_method
)"""


def insert_project(con: sqlite3.Connection, proj: dict) -> int | None:
    """Insert a project row + related persons/keywords/license.
    Returns the new project id, or None if duplicate (IGNORE)."""
    cur = con.cursor()
    cur.execute(INSERT_PROJECT, proj)
    project_id = cur.lastrowid

    if project_id == 0:   # duplicate was ignored
        return None

    # Persons
    for p in proj.get("_persons", []):
        cur.execute(
            "INSERT INTO person_role(project_id, name, role) VALUES (?,?,?)",
            (project_id, p["name"], p["role"])
        )

    # Keywords — one row per raw keyword string
    for kw in proj.get("_keywords", []):
        if kw:  # skip blanks
            cur.execute(
                "INSERT INTO keywords(project_id, keyword) VALUES (?,?)",
                (project_id, str(kw))
            )

    # License
    lic = proj.get("_license")
    if lic:
        cur.execute(
            "INSERT INTO licenses(project_id, license_str) VALUES (?,?)",
            (project_id, str(lic))
        )

    con.commit()
    return project_id


def insert_file(con: sqlite3.Connection, project_id: int,
                file_name: str, status: str = "SUCCEEDED") -> None:
    """Insert a file row linked to a project."""
    ext = file_name.rsplit(".", 1)[-1].lower() if "." in file_name else "unknown"
    con.execute(
        "INSERT INTO files(project_id, file_name, file_type, status) VALUES (?,?,?,?)",
        (project_id, file_name, ext, status)
    )
    con.commit()


print("Insert helpers defined.")

Insert helpers defined.


---
## 5: Download Pipeline

These cells combine search → metadata insert → file download into a single workflow.

**File layout produced:**
```
downloads/
  zenodo/
    <record_id>/
      file1.qdpx
      transcript1.docx
  harvard-dataverse/
    <DVN_ID>/
      file1.qdpx
```

In [22]:
# ── Known open-license identifier sets (for quick filtering) ─────────────────
OPEN_LICENSES = {
    # Creative Commons
    "cc-by-4.0", "cc-by-3.0", "cc-by-2.0",
          "cc-by-sa-4.0", "cc-by-sa-3.0",
    "cc-by-nc-4.0", "cc-by-nc-sa-4.0",
    "cc-by-nd-4.0", "cc-by-nc-nd-4.0",
    "cc0-1.0", "cc-zero",
    # Open Data
    "odc-by", "odbl-1.0", "pddl",
    # Zenodo sometimes uses these strings
    "cc-by", "cc0",
}

SMALL_FILE_EXTENSIONS = [".txt", ".csv", ".json", ".xml"]
MAX_SMALL_FILE_BYTES = 100 * 1024 * 1024  # 100 MB cap for the sample downloader


def is_open_license(lic_str: str | None) -> bool:
    """Return True if the license identifier looks like an open license."""
    if not lic_str:
        return False
    return lic_str.strip().lower() in OPEN_LICENSES


def download_file(url: str, dest_path: Path,
                  chunk_size: int = 1024 * 64,
                  timeout: tuple[int, int] = (10, 30)) -> str:
    """Stream-download a file to dest_path.
    Returns a DOWNLOAD_RESULT status string.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with requests.get(url, stream=True, timeout=timeout) as resp:
            if resp.status_code == 401:
                return "FAILED_LOGIN_REQUIRED"
            if resp.status_code >= 500:
                return "FAILED_SERVER_UNRESPONSIVE"
            resp.raise_for_status()

            content_length = resp.headers.get("content-length")
            if content_length is not None:
                try:
                    length = int(content_length)
                except ValueError:
                    length = None
                if length and length > MAX_SMALL_FILE_BYTES:
                    return "FAILED_TOO_LARGE"

            size = 0
            with open(dest_path, "wb") as fh:
                for chunk in resp.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    fh.write(chunk)
                    size += len(chunk)
                    if size > 4 * 1024 ** 3:   # 4 GB cap
                        return "FAILED_TOO_LARGE"
        return "SUCCEEDED"

    except requests.exceptions.ConnectionError:
        return "FAILED_SERVER_UNRESPONSIVE"
    except requests.exceptions.ReadTimeout:
        return "FAILED_SERVER_UNRESPONSIVE"
    except Exception:
        return "FAILED_SERVER_UNRESPONSIVE"


print("Download helpers defined.")

Download helpers defined.


In [23]:
def run_zenodo_pipeline(query: str, max_pages: int = 3,
                        page_size: int = 10, dry_run: bool = True):
    """Search Zenodo, insert metadata, optionally download files.

    Set dry_run=False to actually download files.
    """
    print(f"\n=== Zenodo pipeline | query='{query}' dry_run={dry_run} ===")
    for page in range(1, max_pages + 1):
        data   = zenodo_search(query, size=page_size, page=page)
        hits   = data.get("hits", {}).get("hits", [])
        if not hits:
            print(f"  Page {page}: no more results.")
            break

        print(f"  Page {page}: {len(hits)} records")
        for rec in hits:
            proj = zenodo_record_to_project(rec, query_string=query)

            # Skip non-open licenses
            if not is_open_license(proj["_license"]):
                print(f"    SKIP [{rec['id']}] license={proj['_license']}")
                continue

            project_id = insert_project(con, proj)
            if project_id is None:
                print(f"    DUP  [{rec['id']}] already in DB")
                continue

            print(f"    NEW  [{rec['id']}] pid={project_id} '{proj['title'][:50]}'")

            # Download each file
            proj_folder = DOWNLOAD_ROOT / proj["download_repository_folder"] \
                                        / proj["download_project_folder"]
            for f in proj["_files"]:
                fname    = f["key"]
                file_url = f.get("links", {}).get("self", "")
                dest     = proj_folder / fname

                if dry_run:
                    status = "SUCCEEDED"   # pretend
                    print(f"      [DRY] {fname}")
                else:
                    status = download_file(file_url, dest)
                    print(f"      {status} {fname}")

                insert_file(con, project_id, fname, status)

            time.sleep(0.5)   # be polite to the API

        time.sleep(1)


# ── Run dry-run with the best Zenodo query ────────────────────────────────────
run_zenodo_pipeline("qdpx", max_pages=1, page_size=5, dry_run=True)


=== Zenodo pipeline | query='qdpx' dry_run=True ===
  Page 1: 5 records
    DUP  [16082705] already in DB
    DUP  [14535515] already in DB
    DUP  [18196304] already in DB
    DUP  [17347915] already in DB
    DUP  [17360366] already in DB


In [24]:
def run_hdv_pipeline(query: str, max_pages: int = 3,
                     page_size: int = 10, dry_run: bool = True):
    """Search Harvard Dataverse, insert metadata, optionally download files."""
    print(f"\n=== Harvard Dataverse pipeline | query='{query}' dry_run={dry_run} ===")
    for page in range(max_pages):
        start = page * page_size
        data  = hdv_search(query, per_page=page_size, start=start)
        items = data.get("data", {}).get("items", [])
        if not items:
            print(f"  Offset {start}: no more results.")
            break

        print(f"  Offset {start}: {len(items)} records")
        for item in items:
            proj = hdv_item_to_project(item, query_string=query)

            project_id = insert_project(con, proj)
            if project_id is None:
                print(f"    DUP  {proj['project_url'][:60]}")
                continue

            print(f"    NEW  pid={project_id} '{proj['title'][:50]}'")

            # Fetch and download files via the native API
            global_id = proj.get("_global_id", "")
            proj_folder = DOWNLOAD_ROOT / proj["download_repository_folder"] \
                                        / proj["download_project_folder"]
            try:
                files_meta = hdv_get_files(global_id)
            except Exception as e:
                print(f"      Could not list files: {e}")
                files_meta = []

            for fm in files_meta:
                df    = fm.get("dataFile", {})
                fname = df.get("filename", "unknown")
                fid   = df.get("id")
                file_url = f"{HDV_API}/access/datafile/{fid}"
                dest  = proj_folder / fname

                if dry_run:
                    status = "SUCCEEDED"
                    print(f"      [DRY] {fname}")
                else:
                    status = download_file(file_url, dest)
                    print(f"      {status} {fname}")

                insert_file(con, project_id, fname, status)

            time.sleep(0.5)
        time.sleep(1)


# ── Run dry-run ───────────────────────────────────────────────────────────────
run_hdv_pipeline("qdpx", max_pages=1, page_size=5, dry_run=True)


=== Harvard Dataverse pipeline | query='qdpx' dry_run=True ===
  Offset 0: 2 records
    DUP  https://doi.org/10.21950/PDWDPG
    DUP  https://doi.org/10.5683/SP3/OOCMMZ


---
## 6: Inspect the Database

In [25]:
# Summary counts across all tables
for table in ["projects", "files", "keywords", "person_role", "licenses"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<20} {n:>5} rows")

  projects                 7 rows
  files                   97 rows
  keywords                31 rows
  person_role             10 rows
  licenses                 5 rows


In [26]:
# Projects per repository
rows = con.execute("""
    SELECT r.name, COUNT(p.id) AS num_projects
    FROM repositories r
    LEFT JOIN projects p ON p.repository_id = r.id
    GROUP BY r.id
""").fetchall()
for row in rows:
    print(f"  {row['name']}: {row['num_projects']} projects")

  Zenodo: 5 projects
  Harvard Dataverse: 2 projects


In [27]:
# File type breakdown
rows = con.execute("""
    SELECT file_type, COUNT(*) AS n
    FROM files
    GROUP BY file_type
    ORDER BY n DESC
    LIMIT 20
""").fetchall()
print("File types found:")
for row in rows:
    print(f"  .{row['file_type']:<15} {row['n']:>4}")

File types found:
  .pdf               43
  .docx              36
  .qdpx               7
  .xlsx               4
  .txt                1
  .tab                1
  .nvp                1
  .mx22               1
  .mx20               1
  .doc                1
  .csv                1


In [28]:
# Download status breakdown
rows = con.execute("""
    SELECT status, COUNT(*) AS n
    FROM files
    GROUP BY status
""").fetchall()
print("Download statuses:")
for row in rows:
    print(f"  {row['status']:<35} {row['n']:>4}")

Download statuses:
  SUCCEEDED                             97


---
## 7: Export Database to CSV (for submission)

The submission form asks for a download link to the database. This cell also exports each table to CSV for easy sharing / peer review.

In [29]:
import csv

EXPORT_DIR = Path("export")
EXPORT_DIR.mkdir(exist_ok=True)

for table in ["repositories", "projects", "files", "keywords",
               "person_role", "licenses"]:
    rows = con.execute(f"SELECT * FROM {table}").fetchall()
    if not rows:
        print(f"  {table}: empty, skipping")
        continue
    out_path = EXPORT_DIR / f"{table}.csv"
    with open(out_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(rows[0].keys())
        writer.writerows(rows)
    print(f"  Exported {table} → {out_path}  ({len(rows)} rows)")

print(f"\nSQLite DB: {DB_PATH.resolve()}")

  Exported repositories → export/repositories.csv  (2 rows)
  Exported projects → export/projects.csv  (7 rows)
  Exported files → export/files.csv  (97 rows)
  Exported keywords → export/keywords.csv  (31 rows)
  Exported person_role → export/person_role.csv  (10 rows)
  Exported licenses → export/licenses.csv  (5 rows)

SQLite DB: /Users/shubhangimore/seeding_qa_analytics_project/qdarchive_metadata.db


---
## 8: Suggested Queries to Post in the Forum

Copy the table below into your forum post.

In [30]:
forum_queries = [
    # (repo, query_string, rationale)
    ("Zenodo",            "qdpx",
     "Direct hit on the REFI-QDA standard file extension — highest signal-to-noise"),
    ("Zenodo",            "mx24 OR mqda OR mx22",
     "MaxQDA project file extensions across recent versions"),
    ("Zenodo",            "nvp OR nvpx OR atlasproj OR hpr7",
     "NVivo + ATLAS.ti extensions"),
    ("Zenodo",            '"qualitative research data" AND (interview OR transcript)',
     "Phrase search for projects explicitly describing qualitative data"),
    ("Zenodo",            'metadata.title:(qualitative) AND metadata.title:(MAXQDA OR NVivo OR "ATLAS.ti")',
     "Title-field targeted search using QDA tool names"),
    ("Zenodo",            '"interview study" AND (transcript OR coding OR thematic)',
     "Broader qualitative methodology signal words"),
    ("Harvard Dataverse", "qdpx",
     "Direct REFI-QDA extension search"),
    ("Harvard Dataverse", 'MAXQDA OR NVivo OR "ATLAS.ti"',
     "QDA tool name search — Dataverse full-text indexes file contents and metadata"),
    ("Harvard Dataverse", '"qualitative" AND "interview" AND "transcript"',
     "Phrase combination targeting interview-based qualitative projects"),
    ("Harvard Dataverse", '"thematic analysis" OR "grounded theory" OR "content analysis"',
     "Qualitative methodology terms that frequently co-occur with QDA files"),
]

print(f"{'Repository':<22} {'Query':<58} Rationale")
print("-" * 120)
for repo, q, why in forum_queries:
    print(f"{repo:<22} {q:<58} {why}")

Repository             Query                                                      Rationale
------------------------------------------------------------------------------------------------------------------------
Zenodo                 qdpx                                                       Direct hit on the REFI-QDA standard file extension — highest signal-to-noise
Zenodo                 mx24 OR mqda OR mx22                                       MaxQDA project file extensions across recent versions
Zenodo                 nvp OR nvpx OR atlasproj OR hpr7                           NVivo + ATLAS.ti extensions
Zenodo                 "qualitative research data" AND (interview OR transcript)  Phrase search for projects explicitly describing qualitative data
Zenodo                 metadata.title:(qualitative) AND metadata.title:(MAXQDA OR NVivo OR "ATLAS.ti") Title-field targeted search using QDA tool names
Zenodo                 "interview study" AND (transcript OR coding OR thematic)   

---
## 9: Known Data Quality Issues to Report

Post these observations in the course forum alongside your queries.

In [31]:
issues = """
=== Data quality observations for forum post ===

1. KEYWORD FORMAT INCONSISTENCY (Zenodo)
   Some records store keywords as a single comma-separated string
   e.g. 'interlanguage pragmatics, EFL learners, scoping review'
   rather than separate values. Stored as-is per the schema rule;
   flagging for the keyword-fixing step.

2. MISSING LICENSE (both repos)
   Some records have no license field at all. These are skipped by
   the pipeline (no license = closed by policy). Counts TBD after
   full crawl.

3. FILES NOT LISTED IN ZENODO SEARCH API
   The /api/records search endpoint does not always include a 'files'
   array for restricted-access records. A separate call to
   /api/records/{id} is needed to confirm the file list.

4. LANGUAGE FIELD ABSENT (Harvard Dataverse)
   The search result item does not expose a language field at this
   level; would require fetching the full metadata block per record.
   Left NULL for now.

5. DUPLICATE RECORDS ACROSS REPOS
   Some datasets are mirrored on both Zenodo and Harvard Dataverse.
   DOI-based deduplication in Part 2 merge step will handle this.
"""
print(issues)


=== Data quality observations for forum post ===

1. KEYWORD FORMAT INCONSISTENCY (Zenodo)
   Some records store keywords as a single comma-separated string
   e.g. 'interlanguage pragmatics, EFL learners, scoping review'
   rather than separate values. Stored as-is per the schema rule;
   flagging for the keyword-fixing step.

2. MISSING LICENSE (both repos)
   Some records have no license field at all. These are skipped by
   the pipeline (no license = closed by policy). Counts TBD after
   full crawl.

3. FILES NOT LISTED IN ZENODO SEARCH API
   The /api/records search endpoint does not always include a 'files'
   array for restricted-access records. A separate call to
   /api/records/{id} is needed to confirm the file list.

4. LANGUAGE FIELD ABSENT (Harvard Dataverse)
   The search result item does not expose a language field at this
   level; would require fetching the full metadata block per record.
   Left NULL for now.

5. DUPLICATE RECORDS ACROSS REPOS
   Some datasets are m

---
## 10: Full Pipeline Run (set `dry_run=False` when ready)

Run all queries across both repos. Safe to re-run — duplicates are ignored via `INSERT OR IGNORE`.

In [32]:
ZENODO_QUERIES = [
    "qdpx",
    "mx24 OR mqda OR mx22",
    "nvp OR nvpx OR atlasproj OR hpr7",
    '"qualitative research data" AND (interview OR transcript)',
    '"interview study" AND (transcript OR coding OR thematic)',
]

HDV_QUERIES = [
    "qdpx",
    'MAXQDA OR NVivo OR "ATLAS.ti"',
    '"qualitative" AND "interview" AND "transcript"',
    '"thematic analysis" OR "grounded theory" OR "content analysis"',
]

DRY_RUN = True   # ← Set to False to actually download files

for q in ZENODO_QUERIES:
    run_zenodo_pipeline(q, max_pages=5, page_size=25, dry_run=DRY_RUN)

for q in HDV_QUERIES:
    run_hdv_pipeline(q, max_pages=5, page_size=25, dry_run=DRY_RUN)

print("\n=== All pipelines complete ===")
for table in ["projects", "files", "keywords", "person_role", "licenses"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<20} {n:>5} rows")


=== Zenodo pipeline | query='qdpx' dry_run=True ===
  Page 1: 5 records
    DUP  [16082705] already in DB
    DUP  [14535515] already in DB
    DUP  [18196304] already in DB
    DUP  [17347915] already in DB
    DUP  [17360366] already in DB
  Page 2: no more results.

=== Zenodo pipeline | query='mx24 OR mqda OR mx22' dry_run=True ===
  Page 1: 3 records
    NEW  [17384976] pid=20 'Qualitative Data of the EduChallenge Project'
      [DRY] Kategoriensystem_FINAL.xlsx
      [DRY] FF1_A-pre_Rohdaten_AnonymUndSortiert.xlsx
      [DRY] Kreuztabellen.zip
      [DRY] FF1_A-post_Rohdaten_AnonymUndSortiert.xlsx
      [DRY] FF1_B_QIA_in_MAXQDA_public.mx24
    NEW  [7886149] pid=21 'Dataset for the Paper: "Security Defect Detection '
      [DRY] dataset.zip
    NEW  [18002402] pid=22 'Replication Package for the Paper: "An Insight int'
      [DRY] replication_package.zip
  Page 2: no more results.

=== Zenodo pipeline | query='nvp OR nvpx OR atlasproj OR hpr7' dry_run=True ===
  Page 1: 13 reco

In [35]:
def run_small_download_sample():
    """Download a very small, GitHub-safe sample of datasets (guarantees at least 1 file)."""

    print("\n=== Running SMALL SAFE DOWNLOAD SAMPLE ===")

    zenodo_query = "qdpx"
    hdv_query    = '"qualitative" AND "interview" AND "transcript"'

    page_size = 2
    sample_max_bytes = 10 * 1024 * 1024  # 10 MB
    sample_extensions = SMALL_FILE_EXTENSIONS + [".xlsx", ".docx", ".pdf"]

    # ─────────────────────────────────────────────────────────────
    # ZENODO
    # ─────────────────────────────────────────────────────────────
    print("\n--- Zenodo sample ---")
    data = zenodo_search(zenodo_query, size=page_size, page=1)
    hits = data.get("hits", {}).get("hits", [])[:2]

    for rec in hits:
        proj = zenodo_record_to_project(rec, query_string=zenodo_query)

        if not is_open_license(proj["_license"]):
            print(f"    SKIP [{rec['id']}] non-open license={proj['_license']}")
            continue

        project_id = insert_project(con, proj)
        if project_id is None:
            print(f"    DUP [{rec['id']}] already in DB")
            continue

        proj_folder = DOWNLOAD_ROOT / proj["download_repository_folder"] / proj["download_project_folder"]

        files = proj["_files"]

        # Prefer small + known extensions
        preferred = [
            f for f in files
            if any(f["key"].lower().endswith(ext) for ext in sample_extensions)
        ]

        # Fallback: ANY file
        candidates = preferred if preferred else files

        if not candidates:
            print(f"    NO_FILES [{rec['id']}]")
            continue

        downloaded = 0

        for f in candidates:
            fname = f["key"]
            file_size = f.get("size")

            if file_size and file_size > sample_max_bytes:
                print(f"    SKIP {fname} ({file_size} bytes) too large")
                continue

            file_url = f.get("links", {}).get("self", "")
            dest = proj_folder / fname

            print(f"    DOWNLOAD {fname} ({file_size or 'unknown'} bytes)")
            status = download_file(file_url, dest)
            print(f"      {status} {fname}")
            insert_file(con, project_id, fname, status)

            downloaded += 1
            if downloaded >= 2:
                break

        if downloaded == 0:
            print(f"    FAILED_TO_DOWNLOAD_ANY [{rec['id']}]")

    # ─────────────────────────────────────────────────────────────
    # HARVARD DATAVERSE
    # ─────────────────────────────────────────────────────────────
    print("\n--- Harvard Dataverse sample ---")
    data = hdv_search(hdv_query, per_page=page_size, start=0)
    items = data.get("data", {}).get("items", [])[:2]

    for item in items:
        proj = hdv_item_to_project(item, query_string=hdv_query)

        project_id = insert_project(con, proj)
        if project_id is None:
            print(f"    DUP {proj['project_url'][:60]}")
            continue

        proj_folder = DOWNLOAD_ROOT / proj["download_repository_folder"] / proj["download_project_folder"]

        try:
            files_meta = hdv_get_files(proj["_global_id"])
        except Exception as e:
            print(f"    COULD_NOT_LIST_FILES {proj['project_url'][:60]}: {e}")
            continue

        if not files_meta:
            print(f"    NO_FILES {proj['project_url'][:60]}")
            continue

        preferred = []
        fallback = []

        for fm in files_meta:
            df = fm.get("dataFile", {})
            fname = df.get("filename", "unknown")
            fid = df.get("id")
            file_size = df.get("filesize")

            if file_size and file_size > sample_max_bytes:
                print(f"    SKIP {fname} ({file_size} bytes) too large")
                continue

            if any(fname.lower().endswith(ext) for ext in sample_extensions):
                preferred.append((fname, fid))
            else:
                fallback.append((fname, fid))

        # Choose preferred if available, otherwise fallback
        candidates = preferred if preferred else fallback

        if not candidates:
            print(f"    NO_VALID_FILES {proj['project_url'][:60]}")
            continue

        downloaded = 0

        for fname, fid in candidates:
            file_url = f"{HDV_API}/access/datafile/{fid}"
            dest = proj_folder / fname

            print(f"    DOWNLOAD {fname}")
            status = download_file(file_url, dest)
            print(f"      {status} {fname}")
            insert_file(con, project_id, fname, status)

            downloaded += 1
            if downloaded >= 2:
                break

        if downloaded == 0:
            print(f"    FAILED_TO_DOWNLOAD_ANY {proj['project_url'][:60]}")

    print("\n=== Small sample download complete ===")

In [36]:
# Run small safe download sample (for GitHub submission)
run_small_download_sample()


=== Running SMALL SAFE DOWNLOAD SAMPLE ===

--- Zenodo sample ---
    DOWNLOAD Theme_year.xlsx (19364 bytes)
      SUCCEEDED Theme_year.xlsx
    DOWNLOAD S2_Table. Data Charting and CCAT Scores of 56 Articles.docx (55604 bytes)
      SUCCEEDED S2_Table. Data Charting and CCAT Scores of 56 Articles.docx
    DOWNLOAD Description of the data set.pdf (368372 bytes)
      SUCCEEDED Description of the data set.pdf
    DOWNLOAD DiscussData-polish-debates-on-higher-education-in-the-sejm-and-party-manifestos-v1.1-description.pdf (55021 bytes)
      SUCCEEDED DiscussData-polish-debates-on-higher-education-in-the-sejm-and-party-manifestos-v1.1-description.pdf

--- Harvard Dataverse sample ---
    NO_FILES https://doi.org/10.3886/ICPSR38668
    NO_FILES https://doi.org/10.3886/ICPSR38668.V1

=== Small sample download complete ===
